In [17]:
import math
import matplotlib
import seaborn
import pandas as pd
import torch
import numpy as np

In [18]:
def simplex(o, c, a, b):
    n, m = a.shape

    if o == 'M':
        c = -c

    # Build tableau: [A | I | b]
    tableau = np.zeros((n + 1, m + n + 1))
    tableau[:n, :m] = a
    tableau[:n, m:m+n] = np.eye(n)
    tableau[:n, -1] = b
    tableau[n, :m] = c

    # Check for infeasibility: any b[i] < 0
    if np.any(b < 0):
        raise ValueError("Problem is infeasible")

    basis = list(range(m, m + n))  # initial basis: slack variables

    while True:
        # Pivot column: most negative in objective row
        pivot_col = np.argmin(tableau[n, :-1])
        if tableau[n, pivot_col] >= 0:
            break  # optimal

        # Minimum ratio test
        ratios = []
        for i in range(n):
            if tableau[i, pivot_col] > 0:
                ratios.append(tableau[i, -1] / tableau[i, pivot_col])
            else:
                ratios.append(np.inf)

        # Unbounded check
        if all(r == np.inf for r in ratios):
            raise ValueError("Problem is unbounded")

        pivot_row = np.argmin(ratios)
        basis[pivot_row] = pivot_col

        # Pivot operation
        tableau[pivot_row] /= tableau[pivot_row, pivot_col]
        for i in range(n + 1):
            if i != pivot_row:
                tableau[i] -= tableau[i, pivot_col] * tableau[pivot_row]

    # Check if basis contains artificial/infeasible variables
    for b_var in basis:
        if b_var >= m + n:
            raise ValueError("Problem is infeasible")

    # Extract solution
    X = np.zeros(m)
    for j in range(m):
        col = tableau[:n, j]
        if np.sum(col == 1) == 1 and np.sum(col == 0) == n - 1:
            X[j] = tableau[np.argmax(col), -1]

    return X

In [19]:
#show LPP#

def show(o, c, a, b):
    
    terms = [f"+ {c[i]}x{i+1}" if c[i] >= 0 else f"- {abs(c[i])}x{i+1}" for i in range(len(c))]
    C = " ".join(terms).lstrip("+ ")
    
    if o == 'm':
        print(f"min z = {C}")
        
    else:
        print(f"max z = {C}")
        
    print("\nSubject to\n")
    
    for i in range(len(b)):
        terms = [f"+ {a[i,j]}x{j+1}" if a[i,j] >= 0 else f"- {abs(a[i,j])}x{j+1}" for j in range(len(c))]
        C = " ".join(terms).lstrip("+ ")
        print(f"{C} <= {b[i]}")

In [20]:
#input#

o = input("M/m")

if o not in {"M","m"}:
    raise ValueError(f"Invalid choice '{o}'. Must be 'M' or 'm'.")

m=int(input("number of variables"))

if m > 20:
    raise ValueError(f"number of variables must be {20} or less. You entered {m}.")

c = np.zeros(m)

for i in range (0,m):
    c[i] = float(input(f"c_{i} = "))

n = int(input("number of constraints"))

if n > 30:
    raise ValueError(f"number of constraints must be {30} or less. You entered {n}.")


a = np.zeros((n,m))
b = np.zeros(n)

for i in range (0,n):
    for j in range (0,m):
        a[i,j] = float(input(f"a_{i},{j} = "))
    b[i] = float(input(f"b_{i} = "))
    

M/m M
number of variables 3
c_0 =  1
c_1 =  2
c_2 =  1
number of constraints 3
a_0,0 =  1
a_0,1 =  -1
a_0,2 =  0
b_0 =  2
a_1,0 =  2
a_1,1 =  0
a_1,2 =  2
b_1 =  4
a_2,0 =  -3
a_2,1 =  1
a_2,2 =  1
b_2 =  11


In [21]:
#print LPP#
show(o,c,a,b)

max z = 1.0x1 + 2.0x2 + 1.0x3

Subject to

1.0x1 - 1.0x2 + 0.0x3 <= 2.0
2.0x1 + 0.0x2 + 2.0x3 <= 4.0
- 3.0x1 + 1.0x2 + 1.0x3 <= 11.0


In [24]:
#solve LPP
X = np.zeros(m)
X = simplex(o, c, a, b)
z = X @ c
print(f"optimal solution is {X}")
print(f"optimal value is {z}")

optimal solution is [ 2. 17.  0.]
optimal value is 36.0
